# 01 — Ingestión: construir el grafo de conocimiento

Este notebook recorre el pipeline de ingestión paso a paso.

**¿Qué ocurre durante la ingestión?**
1. El documento se divide en fragmentos (*chunks*) con solapamiento.
2. Cada chunk recibe un embedding vectorial (guardado en Neo4j para búsqueda por similitud).
3. Un LLM extrae entidades nombradas y relaciones de cada chunk.
4. Todo se escribe en Neo4j como un grafo de propiedades.
5. Las descripciones de entidades y las fuerzas de relación se consolidan entre chunks.

```
Document ──HAS_CHUNK──▶ Chunk ──HAS_ENTITY──▶ Entity
                         │                       │
                    embedding               RELATED_TO
```

**Requisitos:** Neo4j corriendo en `neo4j://localhost:7687`, Ollama en `http://localhost:11434`.

## 1. Configuración

In [1]:
import sys
sys.path.append('..')

from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.ingestion.text_processor import TextProcessor

In [2]:
# Conectar a Neo4j y crear restricciones de esquema e índice vectorial.
# El índice vectorial es lo que hace rápida la búsqueda por similitud más adelante.
neo4j = Neo4jManager()
neo4j.create_constraints()
neo4j.create_vector_index()
print("Neo4j listo.")

Neo4j listo.


## 2. Texto fuente

Usamos una breve biografía de Albert Einstein como documento de ejemplo.
El mismo texto se reutiliza en los tres notebooks de demostración.

In [3]:
sample_text = """
Albert Einstein was a German-born theoretical physicist who is widely held
to be one of the greatest and most influential scientists of all time.
He developed the theory of relativity and made important contributions to
quantum mechanics. Einstein was born in Ulm, Germany in 1879.

Einstein worked at the Swiss Patent Office in Bern from 1902 to 1909.
During this time, he published several groundbreaking papers, including
his work on special relativity in 1905. He later moved to Princeton,
New Jersey, where he worked at the Institute for Advanced Study.

The theory of general relativity, published in 1915, revolutionized our
understanding of gravity and space-time. Einstein received the Nobel Prize
in Physics in 1921 for his explanation of the photoelectric effect.
"""

print(f"Longitud del documento: {len(sample_text)} caracteres")

Longitud del documento: 776 caracteres


## 3. Procesar el documento

`TextProcessor` gestiona el pipeline completo: chunking → embedding → extracción de entidades → almacenamiento en el grafo.

- `chunk_size=200` caracteres por chunk (pequeño para que el LLM tenga contexto enfocado)
- `chunk_overlap=20` caracteres de solapamiento para evitar cortes en medio de frases

In [4]:
processor = TextProcessor(neo4j, chunk_size=200, chunk_overlap=20)

processor.process_document(
    text=sample_text,
    document_id="einstein_bio",
    metadata={"title": "Einstein Biography", "source": "example"}
)

print("\nIngestión completada.")

Consolidating Person: 100%|██████████| 1/1 [00:00<00:00,  1.32it/s]
Consolidating ScientificTheory: 0it [00:00, ?it/s]
Consolidating Publication: 0it [00:00, ?it/s]
Consolidating Institution: 0it [00:00, ?it/s]
Consolidating Location: 100%|██████████| 1/1 [00:00<00:00,  1.63it/s]
Consolidating Award: 0it [00:00, ?it/s]
Consolidating DEVELOPED: 0it [00:00, ?it/s]
Consolidating PUBLISHED: 0it [00:00, ?it/s]
Consolidating CONTRIBUTED_TO: 100%|██████████| 2/2 [00:01<00:00,  1.02it/s]


Ingestión completada.


## 4. Inspeccionar el grafo

Verificamos qué se ha escrito en Neo4j.

In [5]:
# Recuento de nodos por etiqueta
stats = neo4j.execute_query("""
MATCH (n)
RETURN labels(n)[0] AS label, count(*) AS count
ORDER BY count DESC
""")

print("Nodos en el grafo:")
for row in stats:
    print(f"  {row['label']}: {row['count']}")

Nodos en el grafo:
  ScientificTheory: 5
  Location: 5
  Chunk: 4
  Institution: 2
  Person: 2
  ScientificField: 2
  Document: 1
  Publication: 1
  Award: 1


In [6]:
# Nodos por label en el grafo de dominio
node_counts = neo4j.execute_query("""
MATCH (n)
WHERE n:Person OR n:ScientificTheory OR n:Publication
   OR n:Institution OR n:Location OR n:Award
RETURN labels(n)[0] AS label, count(*) AS count
ORDER BY count DESC
""")

print("Nodos de dominio en el grafo:")
for row in node_counts:
    print(f"  {row['label']}: {row['count']}")


Nodos de dominio en el grafo:
  ScientificTheory: 5
  Location: 5
  Institution: 2
  Person: 2
  Publication: 1
  Award: 1


In [7]:
# Relaciones tipadas entre nodos de dominio
rels = neo4j.execute_query("""
MATCH (s)-[r]->(t)
WHERE type(r) IN ['BORN_IN','WORKED_AT','LIVED_IN',
                  'DEVELOPED','PUBLISHED','RECEIVED_AWARD','CONTRIBUTED_TO']
RETURN labels(s)[0] AS src_label, s.name AS source,
       type(r)          AS rel_type,
       labels(t)[0] AS tgt_label, coalesce(t.name, t.title) AS target
ORDER BY rel_type
LIMIT 20
""")

print(f"Relaciones: {len(rels)}\n")
for r in rels:
    print(f"  ({r['src_label']}) {r['source']} "
          f"-[{r['rel_type']}]-> "
          f"({r['tgt_label']}) {r['target']}")


Relaciones: 11

  (Person) Albert: Albert Einstein -[BORN_IN]-> (Location) Germany
  (Person) Albert: Albert Einstein -[BORN_IN]-> (Location) Ulm
  (Person) Albert: Albert Einstein -[CONTRIBUTED_TO]-> (ScientificField) quantum mechanics
  (Person) Albert: Albert Einstein -[CONTRIBUTED_TO]-> (ScientificField) theory of general relativity
  (Person) Albert: Albert Einstein -[DEVELOPED]-> (ScientificTheory) theory of relativity
  (Person) Albert: Albert Einstein -[DEVELOPED]-> (ScientificTheory) quantum mechanics
  (Person) Albert: Albert Einstein -[DEVELOPED]-> (ScientificTheory) special relativity
  (Person) Albert: Einstein -[LIVED_IN]-> (Location) Bern
  (Person) Albert: Albert Einstein -[PUBLISHED]-> (Publication) several groundbreaking
  (Person) Albert: Albert Einstein -[WORKED_AT]-> (Institution) Swiss Patent Office
  (Person) Albert: Albert Einstein -[WORKED_AT]-> (Institution) Institute for Advanced Study


## 5. Limpieza (opcional)

Descomenta la línea siguiente para vaciar la base de datos antes de volver a ejecutar la ingestión.

In [8]:
# neo4j.execute_query("MATCH (n) DETACH DELETE n")
# print("Base de datos vaciada.")

neo4j.close()